In [ ]:
import numpy as np
import xarray as xr
import dask.array as da
from dask_image.ndfilters import median_filter as di_median_filter
from dask.diagnostics import ProgressBar
from pathlib import Path
import atexit
from functools import partial
from scipy import ndimage as nd
from scipy import stats as sts
import warnings
from cogpy.core.utils.grid_neighborhood import make_footprint
from cogpy.core.preprocess.channel_feature import ChannelFeatures, save_features

# -----------------------------------------------------------------------------
# Utility functions
# -----------------------------------------------------------------------------

try:
	script_path = Path(__file__).parent
except NameError:
	script_path = Path.cwd()

# %% neighborhood functions:
def local_robust_zscore(input_arr, footprint=None):
	"""Compute local robust z-score of input array using neighborhood footprint from gneigh."""
	# ndof = input_arr.ndim - 2  # number of leading dimensions before AP, ML
	# footprint_full_shape = (1,) * ndof + gneigh.footprint.shape
	# footprint = gneigh.footprint.reshape(footprint_full_shape)
	if footprint is None:
		footprint = make_footprint(rank=2, connectivity=1, niter=2)

	filter_kwargs = dict(footprint=footprint, mode='constant', cval=np.nan)
	scaled_mad_func = partial(sts.median_abs_deviation, scale='normal', nan_policy='omit')
	local_med = nd.generic_filter(input_arr, function=np.nanmedian, **filter_kwargs)
	local_mad = nd.generic_filter(input_arr, function=scaled_mad_func, **filter_kwargs)
	denom = np.where(local_mad > 0, local_mad, np.nan)
	robust_zscore_center = (input_arr - local_med) / denom
	return robust_zscore_center

def compute_features(
	xsig: xr.DataArray,
	*,
	dim: str = "time",
	window_size: int,
	window_step: int,
	min_periods: int | None = None,
	dtype=np.float32,
	eps: float = 1e-12,
	exact_hurst: bool = False,
	window_dim: str = "window",
) -> xr.Dataset:
	"""
	Compute features over overlapping rolling windows without constructing the
	window dim. Returns values at window centers, strided by `window_step`.

	Features:
	  - deviation         (nanmean over window)
	  - std               (nanstd over window)
	  - amplitude         (nanmax - nanmin over window)
	  - time_derivative   (mean abs derivative over window)
	  - hurst             (≈ 0.5 * log(R/S))
	"""
	assert dim in xsig.dims, f"Dimension '{dim}' not found."
	N = xsig.sizes[dim]
	if min_periods is None:
		min_periods = window_size

	pad = (window_size - 1) // 2
	centers = xr.DataArray(
		np.arange(pad, N - pad, window_step),
		dims=(dim,),
		coords={dim: xsig[dim].isel({dim: slice(pad, N - pad, 1)})[::window_step]},
	)

	roll = xsig.rolling({dim: window_size}, center=True, min_periods=min_periods)

	deviation = roll.mean().astype(dtype).isel({dim: centers})
	std_full = roll.std()
	std = std_full.astype(dtype).isel({dim: centers})
	amplitude = (roll.max() - roll.min()).astype(dtype).isel({dim: centers})

	deriv = xsig.differentiate(dim)
	time_derivative = (
		da.fabs(deriv)
		.rolling({dim: window_size}, center=True, min_periods=min_periods)
		.mean()
		.astype(dtype)
		.isel({dim: centers})
	)

	if not exact_hurst:
		mu = roll.mean()
		y = (xsig - mu).where(xsig.notnull(), 0.0)
		c = y.cumsum(dim=dim)
		c_roll = c.rolling({dim: window_size}, center=True, min_periods=min_periods)
		R = (c_roll.max() - c_roll.min()).astype(dtype)
		R = R.clip(min=eps)
		hurst = (np.log(R / (std_full + eps)) / 2.0).astype(dtype).isel({dim: centers})
	else:
		xwin = roll.construct(window_dim)
		xwin = xwin.isel({dim: centers})
		mean_w = xwin.mean(window_dim, skipna=True)
		std_w = xwin.std(window_dim, skipna=True).astype(dtype)
		ywin = (xwin - mean_w).fillna(0.0)
		zwin = ywin.cumsum(window_dim)
		R_w = (zwin.max(window_dim, skipna=True) - zwin.min(window_dim, skipna=True)).astype(dtype)
		R_w = R_w.clip(min=eps)
		hurst = (np.log(R_w / (std_w + eps)) / 2.0).astype(dtype)

	ds = xr.Dataset(
		dict(
			deviation=deviation,
			std=std,
			amplitude=amplitude,
			time_derivative=time_derivative,
			hurst=hurst,
		)
	)
	for c, v in xsig.coords.items():
		if c != dim:
			ds = ds.assign_coords({c: v})
	ds.attrs = getattr(xsig, "attrs", {})
	return ds

def feature_original_func(sigx: xr.DataArray, window_size: int, window_step: int, zscore: bool) -> xr.Dataset:
	AP = sigx.sizes['AP']
	ML = sigx.sizes['ML']
	fs = sigx.fs
	sigx.attrs['fs'] = fs
	
	print("Setting up ChannelFeatures Extractor ...")
	chfeat = ChannelFeatures(nrows=AP, ncols=ML)

	print("Chunking input data ...")
	chunks = {"time": 16 * 4096, "AP": -1, "ML": -1}
	sigx = sigx.chunk(chunks)
	# take a small slice for testing
	sigx = sigx.isel(time=slice(0, 32 * 4096))

	print("Extracting features ...")
	return chfeat.compute_features(sigx, window_size=window_size, window_step=window_step, zscore=zscore)

In [24]:
# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------

if __name__ == "__main__":
	from dask.distributed import Client, LocalCluster

	# Configure cluster
	cluster = LocalCluster(
		n_workers=2,
		threads_per_worker=2,
		memory_limit="2GB",
		dashboard_address=":8787",
	)
	client = Client(cluster)

	# Register cleanup
	def cleanup():
		print("Cleaning up Dask client and cluster...")
		try:
			client.close()
			cluster.close()
		except Exception:
			pass

	atexit.register(cleanup)

	try:
		print(f"Dashboard available at: {client.dashboard_link}")

		print("Stage 1: Creating test array...")
		large_time_dim = 4_000_000
		shape = (16, 16, large_time_dim)

		x = xr.DataArray(
			da.random.random(shape),
			dims=("AP", "ML", "time"),
			coords={
				"AP": np.arange(shape[0]),
				"ML": np.arange(shape[1]),
				"time": np.arange(shape[2]) * 0.001,
			},
			name="signal",
			attrs={"fs": 651},
		)
		x = x.astype("f4").chunk({"AP": 4, "ML": 4, "time": 200_000})
		print(f"Input array shape: {x.shape}, chunks: {x.chunks}")

		print("Stage 2: Computing features...")
		window_size = 512
		window_step = 128

		# new method
		with ProgressBar():
			x_features = compute_features(
				x,
				dim="time",
				window_size=window_size,
				window_step=window_step,
				min_periods=window_size,
				dtype=np.float32,
				eps=1e-10,
				exact_hurst=False,
			).persist()

		print(f"Feature dataset: {x_features}")
		print(f"Feature array shape: {x_features.deviation.shape}, chunks: {x_features.deviation.chunks}")
		print("Saving raw features to zarr...")
		if (script_path / "test_features.zarr").exists():
			print("test_features.zarr already exists. Skipping save.")
		else:
			x_features.to_zarr(script_path / "test_features.zarr", mode="w")

		print("Stage 3: Computing local robust z-score normalization...")
		foot = make_footprint(rank=2, connectivity=1, niter=2)

		print("Verifying with original method...")
		# old method
		z_ds_original = feature_original_func(x, window_size=window_size, window_step=window_step, zscore=False)

		save_features(z_ds_original, script_path / "test_features_locz_orig.zarr", encoding_per_var=None, consolidate=True, show_progress=True)
		print("All stages complete.")

	except Exception as e:
		print(f"Error during execution: {e}")
		import traceback

		traceback.print_exc()
	finally:
		cleanup()

/storage/share/python/environments/Anaconda3/envs/cogpy/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44051 instead
  warnings.warn(


Dashboard available at: http://127.0.0.1:44051/status
Stage 1: Creating test array...
Input array shape: (16, 16, 4000000), chunks: ((4, 4, 4, 4), (4, 4, 4, 4), (200000, 200000, 200000, 200000, 200000, 200000, 200000, 200000, 200000, 200000, 200000, 200000, 200000, 200000, 200000, 200000, 200000, 200000, 200000, 200000))
Stage 2: Computing features...


/storage/share/python/environments/Anaconda3/envs/cogpy/lib/python3.11/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 32.50 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Feature dataset: <xarray.Dataset> Size: 160MB
Dimensions:          (time: 31247, AP: 16, ML: 16)
Coordinates:
  * time             (time) float64 250kB 0.255 0.383 0.511 ... 4e+03 4e+03
  * AP               (AP) int64 128B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
  * ML               (ML) int64 128B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
Data variables:
    deviation        (AP, ML, time) float32 32MB dask.array<chunksize=(4, 4, 31247), meta=np.ndarray>
    std              (AP, ML, time) float32 32MB dask.array<chunksize=(4, 4, 31247), meta=np.ndarray>
    amplitude        (AP, ML, time) float32 32MB dask.array<chunksize=(4, 4, 31247), meta=np.ndarray>
    time_derivative  (AP, ML, time) float32 32MB dask.array<chunksize=(4, 4, 31247), meta=np.ndarray>
    hurst            (AP, ML, time) float32 32MB dask.array<chunksize=(4, 4, 31247), meta=np.ndarray>
Attributes:
    fs:       651
Feature array shape: (16, 16, 31247), chunks: ((4, 4, 4, 4), (4, 4, 4, 4), (31247,))
Saving raw features to 

2025-09-29 02:35:36,319 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 67d36529cb215fed085cb6e435f89f36 initialized by task ('rechunk-merge-rechunk-transfer-80697e0c76b84334ad0a6db78c056fb2', 0, 0, 2, 0, 0, 1, 3, 0) executed on worker tcp://127.0.0.1:35991
2025-09-29 02:35:47,936 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 67d36529cb215fed085cb6e435f89f36 deactivated due to stimulus 'task-finished-1759106147.9332306'
2025-09-29 02:35:48,038 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle ec77b01909baa33dd78e0b0437b84299 initialized by task ('rechunk-merge-rechunk-transfer-80697e0c76b84334ad0a6db78c056fb2', 0, 0, 0, 0, 0, 1, 0, 0) executed on worker tcp://127.0.0.1:40831
2025-09-29 02:35:52,729 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle ec77b01909baa33dd78e0b0437b84299 deactivated due to stimulus 'task-finished-1759106152.7079241'


...

hurst_exponent: ... 
	


2025-09-29 02:36:47,309 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 67d36529cb215fed085cb6e435f89f36 initialized by task ('rechunk-merge-rechunk-transfer-80697e0c76b84334ad0a6db78c056fb2', 0, 0, 2, 0, 0, 1, 3, 0) executed on worker tcp://127.0.0.1:35991
2025-09-29 02:36:51,143 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 67d36529cb215fed085cb6e435f89f36 deactivated due to stimulus 'task-finished-1759106211.139757'
2025-09-29 02:36:52,140 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle ec77b01909baa33dd78e0b0437b84299 initialized by task ('rechunk-merge-rechunk-transfer-80697e0c76b84334ad0a6db78c056fb2', 0, 0, 0, 0, 0, 1, 0, 0) executed on worker tcp://127.0.0.1:40831
2025-09-29 02:36:55,541 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle ec77b01909baa33dd78e0b0437b84299 deactivated due to stimulus 'task-finished-1759106215.538356'


temporal_mean_laplacian: ... 
	


2025-09-29 02:37:00,271 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 67d36529cb215fed085cb6e435f89f36 initialized by task ('rechunk-merge-rechunk-transfer-80697e0c76b84334ad0a6db78c056fb2', 0, 0, 2, 0, 0, 1, 3, 0) executed on worker tcp://127.0.0.1:35991
2025-09-29 02:37:03,923 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 67d36529cb215fed085cb6e435f89f36 deactivated due to stimulus 'task-finished-1759106223.920676'
2025-09-29 02:37:04,993 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle ec77b01909baa33dd78e0b0437b84299 initialized by task ('rechunk-merge-rechunk-transfer-80697e0c76b84334ad0a6db78c056fb2', 0, 0, 0, 0, 0, 1, 0, 0) executed on worker tcp://127.0.0.1:40831
2025-09-29 02:37:08,111 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle ec77b01909baa33dd78e0b0437b84299 deactivated due to stimulus 'task-finished-1759106228.1079583'


All stages complete.
Cleaning up Dask client and cluster...


In [26]:
x_features = xr.open_zarr("test_features.zarr")

In [32]:
x_orig = xr.open_zarr("test_features_locz_orig.zarr")

In [29]:
foot

array([[False, False,  True, False, False],
       [False,  True,  True,  True, False],
       [ True,  True,  True,  True,  True],
       [False,  True,  True,  True, False],
       [False, False,  True, False, False]])

In [33]:
x_orig['deviation']

<xarray.DataArray 'deviation' (time: 1021, AP: 16, ML: 16)> Size: 1MB
dask.array<open_dataset-deviation, shape=(1021, 16, 16), dtype=float32, chunksize=(1021, 16, 16), chunktype=numpy.ndarray>
Coordinates:
  * AP       (AP) int64 128B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
  * ML       (ML) int64 128B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
  * time     (time) float64 8kB 0.256 0.384 0.512 0.64 ... 130.6 130.7 130.8

In [31]:
x_features['deviation']

<xarray.DataArray 'deviation' (x: 16, y: 16, time: 31247)> Size: 32MB
dask.array<open_dataset-deviation, shape=(16, 16, 31247), dtype=float32, chunksize=(4, 4, 31247), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) float64 250kB 0.255 0.383 0.511 ... 3.999e+03 4e+03 4e+03
  * x        (x) int64 128B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
  * y        (y) int64 128B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15

In [27]:
z_ds = xr.Dataset(
	{
		name: local_robust_zscore(x_features[name], footprint=foot)
		for name in x_features.data_vars
	}
)

print("Saving normalized features to zarr...")
with ProgressBar():
	z_ds.to_zarr(script_path / "test_features_locz.zarr", mode="w")


RuntimeError: footprint.ndim (2) must match len(axes) (3)

In [10]:
feat_ds = xr.open_zarr("test_features.zarr")

In [11]:
feat_ds

<xarray.Dataset> Size: 160MB
Dimensions:          (x: 16, y: 16, time: 31247)
Coordinates:
  * time             (time) float64 250kB 0.255 0.383 0.511 ... 4e+03 4e+03
  * x                (x) int64 128B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
  * y                (y) int64 128B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
Data variables:
    amplitude        (x, y, time) float32 32MB dask.array<chunksize=(4, 4, 31247), meta=np.ndarray>
    deviation        (x, y, time) float32 32MB dask.array<chunksize=(4, 4, 31247), meta=np.ndarray>
    hurst            (x, y, time) float32 32MB dask.array<chunksize=(4, 4, 31247), meta=np.ndarray>
    std              (x, y, time) float32 32MB dask.array<chunksize=(4, 4, 31247), meta=np.ndarray>
    time_derivative  (x, y, time) float32 32MB dask.array<chunksize=(4, 4, 31247), meta=np.ndarray>